In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch

if torch.cuda.is_available():
    print(f"GPU Connected: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not detected. Please check Accelerator settings.")

GPU Connected: Tesla T4


In [3]:
import os
import sys
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
# setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# defining the class mappings, could have been done simply by using image folders but forgot to do that
Class_Mapping = {
    'AnnualCrop': 0, 'Forest': 1, 'HerbaceousVegetation': 2,
    'Highway': 3, 'Industrial': 4, 'Pasture': 5,
    'PermanentCrop': 6, 'Residential': 7, 'River': 8, 'SeaLake': 9
}
# 1. Update the base path variable
base_path = '/kaggle/input/datasets/sakshamgarg2323/eurosat'

train_dir = os.path.join(base_path, 'train', 'train')
test_dir = os.path.join(base_path, 'test_set', 'test_set')



class EuroSATLabeled(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.data = []
        for class_name in os.listdir(self.img_dir):
            class_path = os.path.join(self.img_dir, class_name)
            if os.path.isdir(class_path):
                class_label = Class_Mapping[class_name]
                for img_name in os.listdir(class_path):
                      self.data.append((os.path.join(class_path, img_name), class_label, img_name))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, class_label, img_filename = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, class_label, img_filename

class EuroSATUnlabeled(torch.utils.data.Dataset):
    def __init__(self, img_dir, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms
        self.data = []
        for img_name in os.listdir(self.img_dir):
              self.data.append((os.path.join(self.img_dir, img_name), img_name))

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_path, img_filename = self.data[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transforms:
            img = self.transforms(img)
        return img, img_filename

train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=(0, 360)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Now initialize your datasets
full_train_aug = EuroSATLabeled(img_dir=train_dir, transforms=train_transforms)
full_train_no_aug = EuroSATLabeled(img_dir=train_dir, transforms=test_transforms)
test_dataset = EuroSATUnlabeled(img_dir=test_dir, transforms=test_transforms)
#class for preprocessing of labelled data from train.zip

targets = [item[1] for item in full_train_aug.data]
indices = list(range(len(full_train_aug)))

train_indices, val_indices = train_test_split(
    indices, test_size=0.2, stratify=targets, random_state=42
)

train_dataset = Subset(full_train_aug, train_indices)
val_dataset = Subset(full_train_no_aug, val_indices)
test_dataset = EuroSATUnlabeled(
    img_dir=test_dir,
    transforms=test_transforms
)

batch_size = 256

# --- [UPDATED] Hardware Acceleration for DataLoaders ---
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True,persistent_workers=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Submission samples: {len(test_dataset)}")

import torch.nn.functional as F

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != self.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out
class Bottleneck(nn.Module):
    expansion = 4 # This is crucial: Bottleneck increases out_channels by 4x

    def __init__(self, in_channels, out_channels, stride=1):
        super(Bottleneck, self).__init__()
        # 1x1 conv to compress channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 3x3 conv
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1x1 conv to expand channels back out
        self.conv3 = nn.Conv2d(out_channels, self.expansion * out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion * out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != self.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.linear(out)
        return out

def build_resnet50_64x64(num_classes):
    # Change 'BasicBlock' to 'Bottleneck'
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)

model = build_resnet50_64x64(num_classes=len(Class_Mapping)).to(device)

def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)


model.apply(init_weights)
print("Applied Kaiming Normal initialization.")

num_epochs=70
learning_rate = 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4, amsgrad=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
model = torch.compile(model)
print("Model successfully compiled and optimized.")

scaler = torch.amp.GradScaler('cuda')
cutmix = v2.CutMix(num_classes=len(Class_Mapping), alpha=1.0)

def train_loop(dataloader, model, loss_fn, optimizer, cutmix=None):
    model.train()
    size = len(dataloader.dataset)
    correct = 0
    
    for batch, (X, y, _) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        
        if cutmix is not None:
            X, y = cutmix(X, y)
            
        optimizer.zero_grad()
        
        # --- Run the forward pass in 16-bit Mixed Precision ---
        with torch.amp.autocast('cuda'):
            pred = model(X)
            loss = loss_fn(pred, y)
        
        # --- Scale the loss and update gradients ---
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if batch % 10 == 0: # Prints more often for larger batch sizes
            loss_val = loss.item()
            current = batch * dataloader.batch_size + len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")
            
        if cutmix is not None:
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
        else:
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            
    train_accuracy = correct / size
    print(f"Training Accuracy: {(100*train_accuracy):>0.1f}%")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y, _ in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss


checkpoint_path = '/kaggle/working/EuroSAT_Best_ResNet50.pth'
best_vloss = float('inf')
start_epoch = 0


for t in range(start_epoch, num_epochs):
    print(f"Epoch {t+1}")
    train_loop(train_dataloader, model, loss_fn, optimizer, cutmix=cutmix)
    vloss = test_loop(val_dataloader, model, loss_fn)


    scheduler.step()


    if vloss < best_vloss:
        print(f"Validation loss decreased from {best_vloss:.6f} to {vloss:.6f}. Saving checkpoint...")
        best_vloss = vloss
        torch.save({
            'epoch': t + 1,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_vloss': best_vloss
        }, checkpoint_path)


print("\nLoading the best checkpoint for final evaluation...")
model.load_state_dict(torch.load(checkpoint_path)['model_state'])

print("\n--- Generating Final Validation Report ---")
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for X, y, _ in val_dataloader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        preds = outputs.argmax(dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(Class_Mapping.keys())))

print("\n--- Generating Predictions for Submission ---")
submission_data = []

with torch.no_grad():
    for images, img_names in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        for name, p_idx in zip(img_names, preds):
            submission_data.append({
                'image_id': name,
                'label': p_idx
            })

submission_df = pd.DataFrame(submission_data)
submission_df = submission_df.sort_values(by='image_id').reset_index(drop=True)

csv_filename = '2025MT11352_CNN.csv'
submission_df.to_csv(csv_filename, index=False)

print(f"Submission safely generated and saved to {csv_filename}")


Using device: cuda
Training samples: 18400
Validation samples: 4600
Submission samples: 4000
Applied Kaiming Normal initialization.
Model successfully compiled and optimized.
Epoch 1


W0612 14:17:39.334000 23 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


loss: 2.432972  [  256/18400]
loss: 2.470559  [ 2816/18400]
loss: 1.554056  [ 5376/18400]
loss: 1.738803  [ 7936/18400]
loss: 1.094362  [10496/18400]
loss: 1.423105  [13056/18400]
loss: 1.435218  [15616/18400]
loss: 1.518588  [18176/18400]
Training Accuracy: 54.3%
Validation Error: 
 Accuracy: 67.9%, Avg loss: 0.957636 

Validation loss decreased from inf to 0.957636. Saving checkpoint...
Epoch 2
loss: 1.507531  [  256/18400]
loss: 1.431455  [ 2816/18400]
loss: 1.475776  [ 5376/18400]
loss: 1.135817  [ 7936/18400]
loss: 1.410254  [10496/18400]
loss: 1.236160  [13056/18400]
loss: 1.487103  [15616/18400]
loss: 1.530353  [18176/18400]
Training Accuracy: 67.2%
Validation Error: 
 Accuracy: 71.6%, Avg loss: 0.843233 

Validation loss decreased from 0.957636 to 0.843233. Saving checkpoint...
Epoch 3
loss: 0.892904  [  256/18400]
loss: 1.388674  [ 2816/18400]
loss: 1.412558  [ 5376/18400]
loss: 0.752292  [ 7936/18400]
loss: 1.413656  [10496/18400]
loss: 0.974725  [13056/18400]
loss: 1.297840 